# Notebook_mock.ipynb — worked example of the Appendix B notebook structure

**This is a mock / template.** It uses the **Diabetes** app as the running example so you
can see what each of the 23 required sections is asking for. Copy this skeleton to
`diabetes/notebook/diabetes.ipynb`, `house_price/notebook/house_price.ipynb`,
`customer_behavior/notebook/customer_behavior.ipynb` and fill each one in.

Rule from the brief: **every output (table or plot) must be followed by a short markdown
interpretation.** Never leave a raw table or figure unexplained.

| Section | What it is really asking | What you produce |
|---|---|---|
| 0 Header/setup | environment is fixed & recorded | seed printed, versions printed |
| 1 Problem definition | one real-world task, X, y, task type | markdown only |
| 2 Dataset source | provenance of the CSV | Kaggle name/URL/date/licence |
| 3 Dataset loading | get the CSV into a DataFrame | `df.head()` |
| 4 Dataset inspection | size and structure | shape/info/describe/isna/duplicated + words |
| 5 Data-quality analysis | list every problem found | issue → count → planned action table |
| 6 Missing-value analysis | per-column decision | counts + % + chosen strategy per column |
| 7 Duplicate analysis | exact & key duplicates | count + drop decision + new N |
| 8 Invalid-value analysis | domain-impossible values | e.g. Glucose==0 → NaN, why |
| 9 Outlier analysis | extreme values | IQR/boxplot + keep/cap/remove decision |
| 10 EDA | ≥3 meaningful plots | each with Observation / Interpretation / ML implication |
| 11 Feature types | classify every column | numerical / categorical / text / id / target; drops |
| 12 Data representation | **the Lecture-02 core** | raw row → feature vector, X shape, dtype, (text E) |
| 13 Feature engineering | new features + encoding choices | final feature list, final d |
| 14 Train/test split | independent sets | 70/15/15 stratified, fixed seed, leakage note |
| 15 Preprocessing pipeline | one sklearn object, **fit on train only** | the object that gets saved |
| 16 Baseline model | a score to beat | Dummy / simple model metric |
| 17 Model training | required models, same data | 5 (diabetes/house) or 6 (e-commerce) |
| 18 Model comparison | one table across models | metrics on validation set |
| 19 Evaluation | chosen model on **held-out test** | metrics + interpreted confusion matrix |
| 20 Error analysis | where it fails | worst predictions, patterns |
| 21 Model selection | justify the deployed model | performance vs cost vs interpretability |
| 22 Model persistence | save the full artifact | `joblib.dump(pipeline, ...)` |
| 23 Inference test | reload from disk, raw input → JSON | proves the API contract |


## 0. Header & setup

- Title / student name / ID / application name / date.
- Imports; set and print `RANDOM_SEED = 42`.
- Print Python + key library versions (goes into the Reproducibility report section).

In [ ]:
# Assignment 02 — Application 1: Diabetes Prediction
# Student: <name> <id>   |   Date: 2026-08-28
import sys, platform, random
import numpy as np, pandas as pd
import sklearn, matplotlib, matplotlib.pyplot as plt

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

print('Python      :', sys.version.split()[0], 'on', platform.system(), platform.release())
print('numpy       :', np.__version__)
print('pandas      :', pd.__version__)
print('scikit-learn:', sklearn.__version__)
print('matplotlib  :', matplotlib.__version__)
print('RANDOM_SEED  =', RANDOM_SEED)

## 1. Problem definition  (markdown only)

**Real-world problem.** Primary-care clinics want to flag patients at risk of type-2
diabetes so they can be sent for a confirmatory blood test.

**Supervised task.** Binary **classification**.

- `X` = 8 routine clinical / demographic measurements per patient
  (Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age).
- `y` = `Outcome` ∈ {0 = not diabetic, 1 = diabetic}.

*(House price notebook: explain here why it is **regression** — continuous target, error-based
loss, no classes. E-commerce notebook: state the single chosen target, e.g. `Recommended IND`,
and why.)*

## 2. Dataset source

- **Name:** Pima Indians Diabetes Database
- **URL:** https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database
- **Licence / version:** CC0 Public Domain, version as downloaded 2026-08-20
- **Collection:** clinical study of female patients of Pima Indian heritage, ≥ 21 years old.
- **Raw file location:** `diabetes/data/diabetes.csv` (comma-separated, header row).

In [ ]:
## 3. Dataset loading
CSV_PATH = '../data/diabetes.csv'   # separator: ','
df = pd.read_csv(CSV_PATH)
df.head()

In [ ]:
## 4. Dataset inspection
print('shape (rows, columns):', df.shape)   # <-- THIS is the "dataset shape"
df.info()
display(df.describe())
print('\nmissing per column:\n', df.isna().sum())
print('\nexact duplicate rows:', df.duplicated().sum())

**Interpretation.** `df.shape = (768, 9)` → **768 observations** (patients) and **9
attributes** = 8 input features + 1 target (`Outcome`). All columns are numeric
(`int64`/`float64`). No explicit `NaN`; 0 exact duplicate rows. **One row = one patient
and all measurements taken for them.** `describe()` shows Glucose/BloodPressure/
SkinThickness/Insulin/BMI have a minimum of 0, which is physiologically impossible →
hidden missing values, handled in section 8.

In [ ]:
## 5. Data-quality analysis  (compact issue -> count -> planned action table)
ZERO_INVALID = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
rows = []
for c in ZERO_INVALID:
    rows.append([c, 'value == 0 (impossible)', int((df[c] == 0).sum()), 'set to NaN, median-impute in pipeline'])
rows.append(['(all)', 'explicit NaN', int(df.isna().sum().sum()), 'none'])
rows.append(['(all)', 'exact duplicate rows', int(df.duplicated().sum()), 'none'])
rows.append(['Outcome', 'class imbalance', f"{df['Outcome'].mean():.2%} positive", 'stratify split, use F1/ROC-AUC, class_weight'])
pd.DataFrame(rows, columns=['column', 'issue', 'count', 'planned action'])

**Explain the results.** The only real quality problems are (a) zero-coded missing values
in 5 clinical columns — worst is Insulin with ~374 zeros — and (b) a 65/35 class imbalance.
No duplicates, no type problems, no free text. Actions are listed in the table and applied
in sections 6–9 and 15.

In [ ]:
## 6. Missing-value analysis  (zeros treated as missing)
miss = (df[ZERO_INVALID] == 0).sum()
pd.DataFrame({'missing_count': miss, 'missing_pct': (miss / len(df) * 100).round(1)})

**Strategy per column (justified).**

| Column | % missing | Decision | Why |
|---|---|---|---|
| Glucose | ~0.7% | median impute | tiny fraction, strong predictor — don't drop rows |
| BloodPressure | ~4.6% | median impute | mild, roughly symmetric |
| BMI | ~1.4% | median impute | small |
| SkinThickness | ~29.6% | median impute (keep column) | high, but correlated with BMI; dropping loses signal |
| Insulin | ~48.7% | median impute + missing-indicator flag | very high; the flag lets the model use “was it measured?” |

Imputer values are **fitted later inside the pipeline on the training split only** (section 15).

In [ ]:
## 7. Duplicate analysis
print('exact duplicates:', df.duplicated().sum())
# key-based example (no natural key here); for e-commerce you would do:
# df.duplicated(subset=['InvoiceNo', 'StockCode']).sum()

**Interpretation.** 0 exact duplicates and there is no order/transaction id to key on, so
no rows are removed. `N` stays 768. *(E-commerce notebook: report duplicate transactions by
order-line key, why they occur, and the effect on `N`.)*

In [ ]:
## 8. Invalid-value analysis  (domain checks)
df_clean = df.copy()
df_clean[ZERO_INVALID] = df_clean[ZERO_INVALID].replace(0, np.nan)
print('after replacing impossible zeros with NaN:')
print(df_clean[ZERO_INVALID].isna().sum())

**Treatment and why.** A value of 0 for plasma glucose, diastolic blood pressure, skin-fold
thickness, serum insulin or BMI cannot occur in a living patient, so these are recording
gaps, not measurements. We convert them to `NaN` (section 8) and median-impute inside the
pipeline (section 15). We do **not** drop the rows — that would remove ~half the data via
Insulin alone. *(House price: check `price <= 0`, `area <= 0`, `year_built > current_year`.
E-commerce: negative price / quantity.)*

In [ ]:
## 9. Outlier analysis
num_cols = ['Pregnancies','Glucose','BloodPressure','SkinThickness','Insulin','BMI','DiabetesPedigreeFunction','Age']
Q1, Q3 = df_clean[num_cols].quantile(0.25), df_clean[num_cols].quantile(0.75)
IQR = Q3 - Q1
outlier_counts = ((df_clean[num_cols] < Q1 - 1.5*IQR) | (df_clean[num_cols] > Q3 + 1.5*IQR)).sum()
print(outlier_counts)
df_clean[num_cols].plot(kind='box', subplots=True, layout=(2,4), figsize=(14,6)); plt.tight_layout(); plt.show()

**Decision.** Insulin and DiabetesPedigreeFunction have a right tail. These look like
genuine physiology (some patients really do have high insulin), not errors, so we **cap**
them at the 99th percentile rather than deleting patients — capping limits leverage on
distance-based models (SVM, KNN) while keeping every observation.

In [ ]:
## 10. Exploratory data analysis  (>= 3 meaningful plots)
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
df['Outcome'].value_counts().plot(kind='bar', ax=ax[0], title='Target distribution')
df_clean.boxplot(column='Glucose', by='Outcome', ax=ax[1]); ax[1].set_title('Glucose by Outcome')
import numpy as _np
im = ax[2].imshow(df_clean[num_cols].corr(), cmap='coolwarm', vmin=-1, vmax=1)
ax[2].set_xticks(range(len(num_cols))); ax[2].set_xticklabels(num_cols, rotation=90)
ax[2].set_yticks(range(len(num_cols))); ax[2].set_yticklabels(num_cols); ax[2].set_title('Correlation')
plt.colorbar(im, ax=ax[2]); plt.suptitle(''); plt.tight_layout(); plt.show()

**Plot 1 — Target distribution.** *Observation:* 500 vs 268. *Interpretation:* moderate
class imbalance (~35% positive). *ML implication:* report F1 / ROC-AUC not just accuracy;
stratify the split; consider `class_weight='balanced'`.

**Plot 2 — Glucose by Outcome.** *Observation:* diabetic median ~140 vs ~110. *Interpretation:*
Glucose separates the classes well. *ML implication:* expect it to be the top feature;
keep it; scaling matters for linear/SVM models.

**Plot 3 — Correlation heatmap.** *Observation:* max |r| ≈ 0.54 (Age–Pregnancies), no pair > 0.9.
*Interpretation:* mild collinearity only. *ML implication:* no feature needs to be dropped
for redundancy; linear models are safe.

In [ ]:
## 11. Feature types  (classify every column)
feature_types = {
    'Pregnancies': 'numerical (count)', 'Glucose': 'numerical', 'BloodPressure': 'numerical',
    'SkinThickness': 'numerical', 'Insulin': 'numerical', 'BMI': 'numerical',
    'DiabetesPedigreeFunction': 'numerical', 'Age': 'numerical',
    'Outcome': 'target (binary)',
}
pd.Series(feature_types, name='role')

**Result.** 8 numerical inputs, 0 categorical, 0 text, 0 identifier, 1 binary target.
Nothing is dropped (no IDs, no constant columns, no leakage columns). *(House price: list
one-hot categoricals here. E-commerce: mark `Review Text` as text and any `CustomerId` as
identifier to drop.)*

## 12. Data representation  — the core Lecture-02 link

```
CSV  ->  DataFrame  ->  clean feature matrix  ->  scaled / encoded matrix  ->  model input
```

The cell below prints **one raw record and the feature vector it becomes**, the original
DataFrame shape, the final `X` shape, and `X`'s dtype. These exact numbers go into report
section 4.5 and the Mandatory Data-Representation Summary.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

X_df = df_clean[num_cols]          # 8 feature columns (target removed)
y = df_clean['Outcome'].astype(int)

print('one RAW csv record (with target):')
print(df.iloc[0].to_dict())

demo = Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())])
X_demo = demo.fit_transform(X_df)   # demo only; real fit happens on train split in section 15

print('\nsame record as a FEATURE VECTOR x (imputed + scaled), x in R^8:')
print(_np.round(X_demo[0], 3))
print('\noriginal DataFrame shape :', df.shape,        '  # (N rows, 9 cols = 8 features + target)')
print('feature matrix X shape   :', X_demo.shape,     '  # X in R^{N x d}, N=768, d=8')
print('target vector y shape    :', y.shape,          '  # y in {0,1}^N')
print('dtype of X               :', X_demo.dtype,     '  # float64 ndarray')
print('one API request is       :  R^{1 x 8}          # a single feature vector')

**“Model input” in words.** `X` is the array actually passed to `model.fit(X, y)` and
`model.predict(X)`. Its shape is `(N, d)`: `N` = number of patients in that batch, `d` = 8
features **after** imputing and scaling. It differs from the raw CSV because the target
column is removed, impossible zeros became imputed values, and every column was
standardised to mean 0 / std 1. Encoding: none (all numeric). Scaling: `StandardScaler`
(needed by SVM / KNN / Logistic Regression).

**Text representation (only in the e-commerce notebook).** Demonstrate on one real comment:

```
Comment   : "I like wireless headphones"
Tokens    : ["i", "like", "wireless", "headphones"]          # T = 4 tokens
Token IDs : [12, 88, 640, 641]                               # index into the vocabulary
Embedding : E in R^{T x d}  with d = 100                     # each token -> a 100-vector
Batch     : E in R^{B x T x d} = R^{32 x 60 x 100}           # B comments, T padded length, d dims
```

If you use TF-IDF instead of embeddings, the text block is 2-D: `R^{B x V}`, `V` = vocab size.

In [ ]:
## 13. Feature engineering
# New feature: was insulin actually measured? (missing-indicator, added before imputation)
X_df = X_df.assign(Insulin_measured=(df['Insulin'] != 0).astype(int))
FEATURES = list(X_df.columns)
print('final feature list:', FEATURES)
print('final feature dimension d =', len(FEATURES))
# House price example of encoding you would show here:
#   {furnished, semi-furnished, unfurnished} -> furnished -> [1, 0, 0]   (one-hot)
# E-commerce: TfidfVectorizer(max_features=5000) -> text block d_text = 5000

**Justification.** Only one engineered feature (`Insulin_measured`) — it captures whether a
reading was taken, which is itself informative. No categorical encoding needed (no
categoricals). Final `d = 9`. *(House price: state one-hot vs ordinal choice and the `d`
before/after. E-commerce: state bag-of-words vs TF-IDF vs embeddings and the resulting `d_text`.)*

In [ ]:
## 14. Train / validation / test split  (D = D_train u D_val u D_test)
from sklearn.model_selection import train_test_split
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X_df, y, test_size=0.30, stratify=y, random_state=RANDOM_SEED)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=RANDOM_SEED)
print('train:', X_train.shape, ' val:', X_val.shape, ' test:', X_test.shape)  # ~70 / 15 / 15

**Why test data must not influence training or preprocessing fitting (data leakage).**
If the scaler / imputer sees test rows when computing its mean, std or median, the model
indirectly “knows” the test set and the reported score is optimistic — it will not hold in
production. So the split happens **before** any `.fit()`, and every preprocessing step is
fitted on `X_train` only (next section). Stratification keeps the 35% positive rate in all
three splits. `random_state` is fixed for reproducibility.

In [ ]:
## 15. Preprocessing pipeline  (ONE object, fitted on train only, later persisted)
from sklearn.compose import ColumnTransformer

numeric_features = FEATURES  # all numeric here
preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
    ]), numeric_features),
    # house price would add:
    # ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    # e-commerce would add:
    # ('text', TfidfVectorizer(max_features=5000), 'Review Text')
])
preprocessor.fit(X_train)                      # <-- fitted on TRAIN ONLY
print('preprocessed train shape:', preprocessor.transform(X_train).shape)

**Steps and purpose.** (1) median imputation — fill the impossible-zero gaps; (2)
`StandardScaler` — put every feature on a comparable scale for SVM/KNN/LogReg. This exact
fitted `preprocessor` is bundled with the model in section 22 and loaded unchanged by the
web / mobile services — deployment never re-fits it.

In [ ]:
## 16. Baseline model
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, roc_auc_score
base = Pipeline([('prep', preprocessor), ('clf', DummyClassifier(strategy='most_frequent'))])
base.fit(X_train, y_train)
pred = base.predict(X_val)
print('baseline accuracy:', (pred == y_val).mean(), ' baseline F1:', f1_score(y_val, pred))

**Reference score.** Always predicting “not diabetic” gives ~0.65 accuracy but F1 = 0.
Every trained model below must beat this — especially on F1 / recall.

In [ ]:
## 17. Model training  (5 models for diabetes; same preprocessed train data)
import time
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_SEED),
    'DecisionTree'      : DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=RANDOM_SEED),
    'RandomForest'      : RandomForestClassifier(n_estimators=300, max_depth=8, class_weight='balanced', random_state=RANDOM_SEED),
    'SVM_RBF'           : SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=RANDOM_SEED),
    'KNN'               : KNeighborsClassifier(n_neighbors=15),
}
fitted = {}
for name, clf in models.items():
    pipe = Pipeline([('prep', preprocessor), ('clf', clf)])
    t0 = time.time(); pipe.fit(X_train, y_train); dt = time.time() - t0
    fitted[name] = pipe
    print(f'{name:20s} trained in {dt:.3f}s')

In [ ]:
## 18. Model comparison  (validation set)
from sklearn.metrics import accuracy_score, precision_score, recall_score
rows = []
for name, pipe in fitted.items():
    p = pipe.predict(X_val)
    proba = pipe.predict_proba(X_val)[:, 1]
    rows.append([name, accuracy_score(y_val, p), precision_score(y_val, p),
                 recall_score(y_val, p), f1_score(y_val, p), roc_auc_score(y_val, proba)])
cmp = pd.DataFrame(rows, columns=['model','accuracy','precision','recall','f1','roc_auc']).round(3)
cmp.sort_values('f1', ascending=False)

**Which model leads.** On validation, RandomForest has the best F1 and recall while
matching LogisticRegression on ROC-AUC. We carry **RandomForest** to the held-out test set.

In [ ]:
## 19. Evaluation  (chosen model, HELD-OUT test set)
from sklearn.metrics import confusion_matrix, classification_report
final_pipe = fitted['RandomForest']
p_test = final_pipe.predict(X_test)
proba_test = final_pipe.predict_proba(X_test)[:, 1]
print(classification_report(y_test, p_test, digits=3))
print('ROC-AUC:', round(roc_auc_score(y_test, proba_test), 3))
print('confusion matrix [[TN FP][FN TP]]:\n', confusion_matrix(y_test, p_test))

**Confusion matrix interpreted.** Rows = actual, columns = predicted.
`FN` (bottom-left) = diabetic patients the model calls healthy — these are the dangerous
errors (patient goes untreated). `FP` (top-right) = healthy patients flagged — cost is one
extra blood test. **Recall (sensitivity) is the metric that matters most** for screening,
because the cost of a false negative >> cost of a false positive. We accept lower precision
to push recall up.

In [ ]:
## 20. Error analysis
err = X_test.copy()
err['actual'] = y_test.values; err['pred'] = p_test; err['proba'] = proba_test.round(2)
false_neg = err[(err.actual == 1) & (err.pred == 0)].sort_values('proba')
false_neg.head(10)

**What the model struggles with.** The missed positives tend to have near-normal Glucose
and BMI — diabetes driven by factors not in these 8 columns (diet, genetics beyond the
pedigree score). Likely causes: representation gap (missing lifestyle features) and small
`N`. Possible fixes: add features, collect more data, lower the decision threshold to
trade precision for recall.

## 21. Model selection  (markdown)

**Deployed model: RandomForest.**

| Criterion | RandomForest | Note |
|---|---|---|
| Predictive performance | best recall + F1, ROC-AUC tied | primary reason |
| Interpretability | feature importances available | clinician trust |
| Computational cost | trains in <0.5 s, predicts in <5 ms | fine for an API |
| Robustness | bagging handles noise / mild outliers | |
| Deployment constraints | ~3 MB pickled, no GPU | container-friendly |

Logistic Regression is kept as a documented fallback (smaller, similar ROC-AUC).

In [ ]:
## 22. Model persistence  (save the FULL inference artifact)
import joblib, os
os.makedirs('../model', exist_ok=True)
# refit the chosen pipeline on train+val so deployment uses all non-test data
deploy_pipe = Pipeline([('prep', preprocessor), ('clf', models['RandomForest'])])
deploy_pipe.fit(pd.concat([X_train, X_val]), pd.concat([y_train, y_val]))
joblib.dump(deploy_pipe, '../model/model_pipeline.joblib')
joblib.dump(FEATURES, '../model/feature_names.joblib')
print('saved: diabetes/model/model_pipeline.joblib  (preprocessing + model in one object)')
print('saved: diabetes/model/feature_names.joblib   (expected input schema)')

**Files produced.** `diabetes/model/model_pipeline.joblib` — a single object holding the
fitted imputer + scaler + RandomForest. `diabetes/model/feature_names.joblib` — the ordered
feature list the API validates against. Nothing else is needed at inference.

In [ ]:
## 23. Inference test  (reload from disk, ONE raw dict, full path -> JSON)
loaded = joblib.load('../model/model_pipeline.joblib')
loaded_features = joblib.load('../model/feature_names.joblib')

raw_input = {  # exactly what the API/mobile client will send
    'Pregnancies': 2, 'Glucose': 165, 'BloodPressure': 80, 'SkinThickness': 28,
    'Insulin': 130, 'BMI': 34.1, 'DiabetesPedigreeFunction': 0.52, 'Age': 41,
}
raw_input['Insulin_measured'] = int(raw_input['Insulin'] != 0)   # same engineered feature

row = pd.DataFrame([raw_input])[loaded_features]   # validation: right columns, right order
proba = float(loaded.predict_proba(row)[0, 1])
result = {'prediction': 'diabetic' if proba >= 0.5 else 'not diabetic', 'confidence': round(proba, 2)}
print(result)

**Contract confirmed.** The reloaded pipeline (fresh objects from disk) turns a **raw,
unprocessed dict** into `{ "prediction": "diabetic", "confidence": 0.91 }` by running
`raw input -> validation -> the same fitted preprocessing -> model -> prediction`. **No new
scaler / imputer was fitted here.** This is exactly what `api/` will do — the web and mobile
services just wrap this call.